# Single-CPU Playout Campaign Analysis

Analysis of autonomous performance-search results from `results.tsv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("results.tsv", sep="\t")
df["median_speedup_pct"] = pd.to_numeric(df["median_speedup_pct"], errors="coerce")
df["playouts_per_cpu_second"] = pd.to_numeric(df["playouts_per_cpu_second"], errors="coerce")
df["status"] = df["status"].fillna("").astype(str).str.strip().str.upper()

print(f"Total experiments: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
counts = df["status"].value_counts()
print("Experiment outcomes:")
print(counts.to_string())

n_keep = counts.get("KEEP", 0)
n_discard = counts.get("DISCARD", 0)
n_fail = len(df) - n_keep - n_discard
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")
print(f"Failure count: {n_fail}")

In [ ]:
kept = df[df["status"] == "KEEP"].copy()
print(f"KEPT experiments ({len(kept)} total):\n")
for i, row in kept.iterrows():
    speedup = row["median_speedup_pct"]
    pps = row["playouts_per_cpu_second"]
    desc = row["description"]
    print(f"  #{i:3d}  speedup={speedup:+.4f}%  pps={pps:.2f}  {desc}")

## Throughput Over Time

Track how accepted throughput evolves as experiments progress. The running maximum shows the current frontier.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

valid = df[df["status"].isin(["KEEP", "DISCARD"])].copy().reset_index(drop=True)
kept_v = valid[valid["status"] == "KEEP"]
disc = valid[valid["status"] == "DISCARD"]

ax.scatter(disc.index, disc["playouts_per_cpu_second"],
           c="#cccccc", s=18, alpha=0.6, zorder=2, label="Discarded")
ax.scatter(kept_v.index, kept_v["playouts_per_cpu_second"],
           c="#2ecc71", s=55, zorder=4, label="Kept", edgecolors="black", linewidths=0.5)

if not kept_v.empty:
    running_max = kept_v["playouts_per_cpu_second"].cummax()
    ax.step(kept_v.index, running_max, where="post", color="#27ae60",
            linewidth=2, alpha=0.8, zorder=3, label="Running best")

for idx, row in kept_v.iterrows():
    desc = str(row["description"]).strip()
    if len(desc) > 28:
        desc = desc[:25] + "..."
    ax.annotate(desc, (idx, row["playouts_per_cpu_second"]), xytext=(5, 6),
                textcoords="offset points", fontsize=8, alpha=0.9)

ax.set_title("Playout Throughput Over Experiments")
ax.set_xlabel("Experiment Index")
ax.set_ylabel("Playouts Per CPU Second")
ax.grid(alpha=0.2)
ax.legend()
plt.show()

## Summary Statistics

In [ ]:
kept = df[df["status"] == "KEEP"].copy()
if kept.empty:
    print("No kept experiments yet.")
else:
    baseline_row = kept.iloc[0]
    best_row = kept.loc[kept["playouts_per_cpu_second"].idxmax()]
    baseline_pps = baseline_row["playouts_per_cpu_second"]
    best_pps = best_row["playouts_per_cpu_second"]
    if baseline_pps > 0:
        total_gain_pct = (best_pps - baseline_pps) / baseline_pps * 100.0
    else:
        total_gain_pct = float("nan")

    print(f"Baseline playouts/s: {baseline_pps:.2f}")
    print(f"Best playouts/s:     {best_pps:.2f}")
    print(f"Total gain:          {total_gain_pct:.2f}%")
    print(f"Best experiment:     {best_row['description']}")
    print()
    print("Kept frontier:")
    for i, row in kept.reset_index().iterrows():
        print(
            f"  Experiment #{row['index']:3d}: "
            f"pps={row['playouts_per_cpu_second']:.2f} "
            f"speedup={row['median_speedup_pct']:+.4f}% "
            f"{row['description']}"
        )

## Top Hits

In [ ]:
kept = df[df["status"] == "KEEP"].copy().reset_index(drop=True)
if len(kept) <= 1:
    print("Not enough kept experiments for hit ranking.")
else:
    kept["prev_pps"] = kept["playouts_per_cpu_second"].shift(1)
    kept["delta_pps"] = kept["playouts_per_cpu_second"] - kept["prev_pps"]
    hits = kept.iloc[1:].copy().sort_values("delta_pps", ascending=False)

    print(f"{'Rank':>4}  {'Delta PPS':>10}  {'Speedup %':>10}  Description")
    print("-" * 96)
    for rank, (_, row) in enumerate(hits.iterrows(), 1):
        print(
            f"{rank:4d}  {row['delta_pps']:+10.2f}  {row['median_speedup_pct']:+10.4f}  {row['description']}"
        )

    total_delta = kept.iloc[-1]["playouts_per_cpu_second"] - kept.iloc[0]["playouts_per_cpu_second"]
    print(f"\n{'':>4}  {total_delta:+10.2f}  {'':>10}  TOTAL gain over first kept baseline")